# Advanced EDA (Chronological Notebook)

This notebook reproduces the same exploration intent as the old script, but in a **normal, chronological EDA workflow**: load data, inspect class/protocol distributions, check split shift, then inspect feature separability.

It does **not** auto-generate a final report file.

## 0) Setup
Set paths, seed, and import libraries.

In [ ]:
from __future__ import annotations

import csv
import math
import random
import statistics
from array import array
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 20260305
SAMPLE_RATE = 0.006

random.seed(RANDOM_SEED)

base_candidates = [
    Path('/home/capstone15/data/ciciomt2024/merged'),
    Path('data/merged'),
]

BASE = next((b for b in base_candidates if (b / 'metadata_train.csv').exists()), base_candidates[1])
TRAIN_CSV = BASE / 'metadata_train.csv'
TEST_CSV = BASE / 'metadata_test.csv'

print('Using BASE:', BASE)
for p in [TRAIN_CSV, TEST_CSV]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

## 1) Quick schema inspection
Start by checking columns and identifying metadata vs feature columns.

In [ ]:
METADATA_COLUMNS = [
    'source_relpath', 'source_filename', 'source_modality', 'source_group',
    'source_split_folder', 'assigned_split', 'split_strategy', 'sample_name',
    'sample_core_name', 'protocol_scope', 'protocol_hint', 'device', 'scenario',
    'attack_name', 'attack_family', 'attack_variant', 'is_attack', 'is_benign',
    'label', 'source_row_index',
]

with TRAIN_CSV.open('r', newline='', encoding='utf-8', errors='replace') as f:
    header = next(csv.reader(f))

feature_columns = [c for c in header if c not in METADATA_COLUMNS]

print('Total columns:', len(header))
print('Metadata columns found:', len([c for c in header if c in METADATA_COLUMNS]))
print('Feature columns:', len(feature_columns))
print('First 10 feature columns:', feature_columns[:10])

In [ ]:
train_preview = pd.read_csv(TRAIN_CSV, nrows=5)
test_preview = pd.read_csv(TEST_CSV, nrows=5)

display(train_preview.head())
display(test_preview.head())

## 2) Full pass over train+test for high-level distribution counts
This pass computes global counts and also stores a small sampled subset for deeper feature analysis.

In [ ]:
def safe_float(raw: str):
    txt = (raw or '').strip()
    if txt == '':
        return 0.0, True
    try:
        return float(txt), False
    except ValueError:
        return 0.0, True

def maybe_int(raw, default=0):
    txt = str(raw).strip() if raw is not None else ''
    if txt == '':
        return default
    try:
        return int(float(txt))
    except ValueError:
        return default

label_counts = Counter()
label_by_split = Counter()
protocol_counts = Counter()
protocol_by_label = Counter()
protocol_by_split = Counter()
protocol_by_split_label = Counter()
family_counts = Counter()
attack_name_counts = Counter()
scenario_counts = Counter()
source_file_counts = Counter()

sampled_feature_values = {
    0: {f: array('f') for f in feature_columns},
    1: {f: array('f') for f in feature_columns},
}
sampled_missing_counts = {
    0: {f: 0 for f in feature_columns},
    1: {f: 0 for f in feature_columns},
}
sampled_zero_counts = {
    0: {f: 0 for f in feature_columns},
    1: {f: 0 for f in feature_columns},
}
sampled_row_count = Counter()

total_rows = 0
for split_name, csv_path in [('train', TRAIN_CSV), ('test', TEST_CSV)]:
    with csv_path.open('r', newline='', encoding='utf-8', errors='replace') as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_rows += 1

            label = maybe_int(row.get('label', '0'), default=0)
            label = 1 if label == 1 else 0

            protocol = (row.get('protocol_hint', '') or 'unknown').strip() or 'unknown'
            family = (row.get('attack_family', '') or 'unknown').strip() or 'unknown'
            attack_name = (row.get('attack_name', '') or 'unknown').strip() or 'unknown'
            scenario = (row.get('scenario', '') or 'unknown').strip() or 'unknown'
            source_relpath = (row.get('source_relpath', '') or 'unknown').strip() or 'unknown'

            label_counts[label] += 1
            label_by_split[(split_name, label)] += 1
            protocol_counts[protocol] += 1
            protocol_by_label[(protocol, label)] += 1
            protocol_by_split[(split_name, protocol)] += 1
            protocol_by_split_label[(split_name, protocol, label)] += 1
            family_counts[family] += 1
            scenario_counts[scenario] += 1
            source_file_counts[source_relpath] += 1
            if label == 1:
                attack_name_counts[attack_name] += 1

            if random.random() < SAMPLE_RATE:
                sampled_row_count[label] += 1
                for fcol in feature_columns:
                    val, missing = safe_float(row.get(fcol, ''))
                    sampled_feature_values[label][fcol].append(val)
                    if missing:
                        sampled_missing_counts[label][fcol] += 1
                    if val == 0.0:
                        sampled_zero_counts[label][fcol] += 1

print('Total rows:', total_rows)
print('Sampled benign rows:', sampled_row_count[0])
print('Sampled attack rows:', sampled_row_count[1])

## 3) Class balance

In [ ]:
benign_total = label_counts[0]
attack_total = label_counts[1]

class_df = pd.DataFrame({
    'label': ['benign', 'attack'],
    'rows': [benign_total, attack_total],
})
class_df['share'] = class_df['rows'] / max(1, class_df['rows'].sum())
display(class_df)

plt.figure(figsize=(6,4))
plt.bar(class_df['label'], class_df['rows'])
plt.title('Class Balance (All Rows)')
plt.ylabel('Rows')
plt.show()

## 4) Label balance by split (train vs test)

In [ ]:
split_rows = []
for split in ['train', 'test']:
    b = label_by_split[(split, 0)]
    a = label_by_split[(split, 1)]
    t = b + a
    split_rows.append({
        'split': split,
        'rows': t,
        'benign_rows': b,
        'attack_rows': a,
        'benign_ratio': b / max(1, t),
        'attack_ratio': a / max(1, t),
    })
split_df = pd.DataFrame(split_rows)
display(split_df)

## 5) Test-set protocol composition by label

In [ ]:
test_protocol_rows = []
for protocol in sorted(protocol_counts, key=lambda p: protocol_by_split[('test', p)], reverse=True):
    benign_n = protocol_by_split_label[('test', protocol, 0)]
    attack_n = protocol_by_split_label[('test', protocol, 1)]
    total_n = benign_n + attack_n
    if total_n == 0:
        continue

    test_protocol_rows.append({
        'protocol_hint': protocol,
        'test_rows': total_n,
        'benign_test_rows': benign_n,
        'attack_test_rows': attack_n,
        'benign_ratio': benign_n / max(1, total_n),
        'attack_ratio': attack_n / max(1, total_n),
    })

test_protocol_df = pd.DataFrame(test_protocol_rows)
display(test_protocol_df)

## 6) Protocol composition

In [ ]:
protocol_total = sum(protocol_counts.values())
protocol_rows = []
for protocol, total_n in protocol_counts.most_common():
    protocol_rows.append({
        'protocol_hint': protocol,
        'rows': total_n,
        'share': total_n / max(1, protocol_total),
    })
protocol_df = pd.DataFrame(protocol_rows)
display(protocol_df.head(15))

top_n = min(10, len(protocol_df))
top_proto = protocol_df.head(top_n).copy()
x = np.arange(top_n)

plt.figure(figsize=(12,5))
plt.bar(x, top_proto['rows'])
plt.xticks(x, top_proto['protocol_hint'], rotation=45, ha='right')
plt.title('Top Protocols by Total Row Count')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

## 7) Train-test protocol shift

In [ ]:
train_total = sum(protocol_by_split[('train', p)] for p in protocol_counts)
test_total = sum(protocol_by_split[('test', p)] for p in protocol_counts)

shift_rows = []
for p in protocol_counts:
    tr = protocol_by_split[('train', p)]
    te = protocol_by_split[('test', p)]
    tr_share = tr / max(1, train_total)
    te_share = te / max(1, test_total)
    shift_rows.append({
        'protocol_hint': p,
        'train_rows': tr,
        'test_rows': te,
        'train_share': tr_share,
        'test_share': te_share,
        'share_diff_test_minus_train': te_share - tr_share,
    })

shift_df = pd.DataFrame(shift_rows).sort_values(
    'share_diff_test_minus_train', key=lambda s: s.abs(), ascending=False
)
display(shift_df.head(15))

## 8) Attack families and attack names

In [ ]:
family_df = pd.DataFrame(family_counts.most_common(), columns=['attack_family', 'rows'])
family_df['share'] = family_df['rows'] / max(1, family_df['rows'].sum())
display(family_df.head(15))

attack_name_df = pd.DataFrame(attack_name_counts.most_common(25), columns=['attack_name', 'rows'])
display(attack_name_df.head(25))

top_k = min(12, len(family_df))
plot_df = family_df.head(top_k)
plt.figure(figsize=(12,4.8))
plt.bar(plot_df['attack_family'], plot_df['rows'])
plt.xticks(rotation=45, ha='right')
plt.title('Top Attack Families')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

## 9) Sample-based feature separability (benign vs attack)

In [ ]:
def pooled_std(vals0, vals1):
    n0, n1 = len(vals0), len(vals1)
    if n0 < 2 or n1 < 2:
        return 0.0
    v0 = statistics.pvariance(vals0)
    v1 = statistics.pvariance(vals1)
    den = (n0 + n1 - 2)
    if den <= 0:
        return 0.0
    return math.sqrt((((n0 - 1) * v0) + ((n1 - 1) * v1)) / den)

feature_rows = []
for fcol in feature_columns:
    b_vals = sampled_feature_values[0][fcol]
    a_vals = sampled_feature_values[1][fcol]
    if len(b_vals) < 10 or len(a_vals) < 10:
        continue

    b = b_vals.tolist()
    a = a_vals.tolist()

    m0 = statistics.fmean(b)
    m1 = statistics.fmean(a)
    s0 = statistics.pstdev(b) if len(b) > 1 else 0.0
    s1 = statistics.pstdev(a) if len(a) > 1 else 0.0
    ps = pooled_std(b, a)
    d = (m1 - m0) / ps if ps > 0 else 0.0

    n0, n1 = len(b), len(a)
    z0 = sampled_zero_counts[0][fcol]
    z1 = sampled_zero_counts[1][fcol]
    miss0 = sampled_missing_counts[0][fcol]
    miss1 = sampled_missing_counts[1][fcol]

    feature_rows.append({
        'feature': fcol,
        'benign_sample_n': n0,
        'attack_sample_n': n1,
        'benign_mean': m0,
        'attack_mean': m1,
        'benign_std': s0,
        'attack_std': s1,
        'cohens_d_attack_minus_benign': d,
        'benign_zero_rate': z0 / max(1, n0),
        'attack_zero_rate': z1 / max(1, n1),
        'benign_missing_rate': miss0 / max(1, n0),
        'attack_missing_rate': miss1 / max(1, n1),
    })

feature_sep_df = pd.DataFrame(feature_rows)
feature_sep_df = feature_sep_df.sort_values(
    'cohens_d_attack_minus_benign', key=lambda s: s.abs(), ascending=False
)
display(feature_sep_df.head(20))

## 10) Sample-based outlier check in the data (IQR rule)

In [ ]:
outlier_rows = []
for fcol in feature_columns:
    combined = sampled_feature_values[0][fcol].tolist() + sampled_feature_values[1][fcol].tolist()
    if len(combined) < 20:
        continue

    q1 = float(np.quantile(combined, 0.25))
    q3 = float(np.quantile(combined, 0.75))
    iqr = q3 - q1
    if iqr <= 0:
        continue

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_count = sum(1 for v in combined if v < lower_fence or v > upper_fence)

    outlier_rows.append({
        'feature': fcol,
        'combined_sample_n': len(combined),
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_fence': lower_fence,
        'upper_fence': upper_fence,
        'outlier_count': outlier_count,
        'outlier_rate': outlier_count / max(1, len(combined)),
        'min_sampled_value': min(combined),
        'max_sampled_value': max(combined),
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values(
    ['outlier_rate', 'outlier_count'], ascending=False
)
display(outlier_df.head(20))

plot_n = min(10, len(outlier_df))
if plot_n > 0:
    plot_df = outlier_df.head(plot_n).iloc[::-1]
    plt.figure(figsize=(10,5))
    plt.barh(plot_df['feature'], plot_df['outlier_rate'])
    plt.title('Top Features by Sampled Outlier Rate (IQR Rule)')
    plt.xlabel('Outlier rate')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 11) Distribution overlays for top separability features

In [ ]:
top_features = feature_sep_df['feature'].head(3).tolist()

for feat in top_features:
    b = sampled_feature_values[0][feat].tolist()
    a = sampled_feature_values[1][feat].tolist()

    plt.figure(figsize=(8,4.5))
    plt.hist(b, bins=36, alpha=0.45, label='Benign')
    plt.hist(a, bins=36, alpha=0.45, label='Attack')
    plt.title(f'Distribution Overlay: {feat}')
    plt.xlabel(feat)
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12) Correlation check among top features

In [ ]:
top8 = feature_sep_df['feature'].head(8).tolist()
corr_data = {}
for feat in top8:
    corr_data[feat] = sampled_feature_values[0][feat].tolist() + sampled_feature_values[1][feat].tolist()

corr_df = pd.DataFrame(corr_data).corr()
display(corr_df)

plt.figure(figsize=(7,6))
im = plt.imshow(corr_df.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title('Top-8 Feature Correlation Matrix (Sampled)')
plt.tight_layout()
plt.show()

## 13) Plain-English takeaways
Use this section as your EDA conclusion before modeling:

- Confirm whether class imbalance is severe enough to require threshold-centric evaluation.
- Check protocol dominance and train/test protocol shift before making global claims.
- Identify the most discriminative features and verify they are plausible flow features.
- Check whether any continuous features have extreme tails or high outlier rates before scaling or modeling.
- Use these findings to define threshold policy, robustness priorities, and explainability focus.